# Week 08 · Chapter 07 — Deep Learning & Generative AI

README.md의 순서와 동일하게 구성한 Google Colab Notebook. 각 Chapter는 독립적인 코드 셀로 실행 가능함. **7.3과 7.4는 처음 실행할 때 공개 모델을 다운로드**하므로 인터넷 연결이 필요함.

## 실행 환경 준비

Colab 기본 환경에 `torch`, `torchvision`이 설치되어 있는지 확인함. 없는 경우 별도 셀에서 `%pip install -q torch torchvision`을 실행함.

In [ ]:
%pip install -q numpy matplotlib scikit-learn pillow transformers

In [ ]:
import torch, torchvision, sklearn, transformers
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("sklearn:", sklearn.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

## 7.1. Introduction to Deep Learning

고정된 가중치를 이용한 순전파 계산. **이 예제는 학습이 아닌 계산 시연**임.

In [ ]:
"""7.1: 작은 인공신경망의 순전파 계산 — NumPy만 사용."""
import numpy as np

# 두 개의 특징: [공부 시간(정규화), 수면 시간(정규화)]
x = np.array([0.6, 0.8])

# 입력 2개 → 은닉 노드 3개 → 출력 1개
W1 = np.array([[0.4, -0.3, 0.7], [0.2, 0.6, -0.1]])
b1 = np.array([0.1, 0.0, 0.2])
W2 = np.array([[0.5], [-0.4], [0.8]])
b2 = np.array([0.1])

hidden = np.maximum(x @ W1 + b1, 0)  # ReLU: 음수를 0으로
output = hidden @ W2 + b2

print("입력층:", x, "shape:", x.shape)
print("은닉층:", np.round(hidden, 3), "shape:", hidden.shape)
print("출력층:", np.round(output, 3), "shape:", output.shape)
print("주의: 학습하지 않은 임의의 가중치이므로 실제 시험 점수 예측이 아님")


## 7.2. Neural Networks & Model Training

8×8 손글씨 숫자 데이터를 직접 학습하고 테스트 정확도·Loss 그래프 확인.

In [ ]:
"""7.2: PyTorch로 8×8 손글씨 숫자(0~9) 분류 신경망 학습.

별도 CSV 불필요. CPU 실행 가능. 실행하면 학습 곡선과 예측 이미지가 assets/에 저장됨.
"""
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
EPOCHS = 15
LEARNING_RATE = 0.01
BATCH_SIZE = 64
HIDDEN_UNITS = 64

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 2))

out_dir = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
out_dir.mkdir(parents=True, exist_ok=True)

digits = load_digits()
X = (digits.data / 16.0).astype(np.float32)   # 8×8 픽셀 64개, 값 0~1
y = digits.target.astype(np.int64)            # 실제 숫자 0~9

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True,
)

# 입력 64개(8×8) → 은닉층 64개 → 출력 10개(숫자 0~9)
model = nn.Sequential(
    nn.Linear(64, HIDDEN_UNITS),
    nn.ReLU(),
    nn.Linear(HIDDEN_UNITS, 10),
)
criterion = nn.CrossEntropyLoss()  # 숫자 종류를 구분하는 분류 손실함수
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_history = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
    avg_loss = total_loss / len(X_train)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Train Loss: {avg_loss:.4f}")

# 학습 완료 후 테스트 데이터에 대한 예측
model.eval()
with torch.no_grad():
    test_logits = model(torch.from_numpy(X_test))
    predictions = test_logits.argmax(dim=1).numpy()
accuracy = (predictions == y_test).mean()
print(f"Test Accuracy: {accuracy:.1%} ({(predictions == y_test).sum()}/{len(y_test)})")

# 시각화 1 — 학습이 진행되는 동안 Loss 변화
plt.figure(figsize=(9, 5.3), dpi=160)
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o", linewidth=2)
plt.title("Training Loss by Epoch")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.xticks(range(1, EPOCHS + 1))
plt.grid(alpha=0.28)
plt.tight_layout()
plt.savefig(out_dir / "7_2_training_curve.png", facecolor="white", dpi=160)
plt.show()

# 시각화 2 — 처음 12개 테스트 이미지의 예측 결과
fig, axes = plt.subplots(3, 4, figsize=(10, 8), dpi=160)
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r", interpolation="nearest")
    correct = predictions[i] == y_test[i]
    ax.set_title(f"True: {y_test[i]} / Pred: {predictions[i]}" + ("" if correct else " *"), fontsize=9)
    ax.axis("off")
fig.suptitle("Handwritten Digit Classification: Test Samples", fontsize=14)
fig.tight_layout(pad=2.2, h_pad=2.4, w_pad=1.0)
fig.savefig(out_dir / "7_2_digits_predictions.png", facecolor="white", dpi=160)
plt.show()


## 7.3. Computer Vision & Image Recognition

MobileNetV3-Small 사전 학습 모델의 사진 인식. 샘플 이미지와 모델 가중치 다운로드 필요. `UPLOAD_OWN_IMAGE=True`면 사진 직접 업로드.

In [ ]:
"""7.3: ImageNet 사전 학습 MobileNetV3로 사진 분류.

처음 실행할 때 PyTorch Hub 예제 이미지/모델 가중치를 내려받으므로 인터넷 필요.
UPLOAD_OWN_IMAGE=True로 변경하면 Colab에서 직접 이미지를 업로드할 수 있음.
"""
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

UPLOAD_OWN_IMAGE = False
SAMPLE_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

out_dir = (Path(__file__).resolve().parents[1] / "assets") if "__file__" in globals() else Path("assets")
out_dir.mkdir(parents=True, exist_ok=True)

if UPLOAD_OWN_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("이미지 파일을 업로드해야 합니다.")
    img_path = Path(next(iter(uploaded)))
else:
    img_path = out_dir / "sample_dog.jpg"
    if not img_path.is_file():
        print("PyTorch Hub 공개 예제 이미지를 다운로드합니다.")
        urlretrieve(SAMPLE_IMAGE_URL, img_path)

image = Image.open(img_path).convert("RGB")
weights = MobileNet_V3_Small_Weights.DEFAULT
preprocess = weights.transforms()    # 모델에 맞춘 Resize, Crop, Normalize
model = mobilenet_v3_small(weights=weights)
model.eval()                          # 추론 모드

with torch.inference_mode():
    logits = model(preprocess(image).unsqueeze(0))
    probs = logits.softmax(dim=1)[0] # 1000개 클래스에 대한 점수 분포
    values, indices = torch.topk(probs, 5)

labels = weights.meta["categories"]
print("Top 5 predictions:")
for rank, (idx, value) in enumerate(zip(indices.tolist(), values.tolist()), 1):
    print(f"{rank}. {labels[idx]}: {value:.1%}")

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11.6, 4.4), dpi=150)
ax_img.imshow(image)
ax_img.set_title("Input Image")
ax_img.axis("off")
names = [labels[i] for i in indices.tolist()]
ax_bar.barh(names[::-1], [x * 100 for x in values.tolist()[::-1]])
ax_bar.set_xlabel("Softmax Score (%)")
ax_bar.set_title("Top 5 Predicted Classes")
ax_bar.set_xlim(0, 100)
fig.tight_layout()
fig.savefig(out_dir / "7_3_image_prediction.png", facecolor="white", dpi=160)
plt.show()
print("모델의 점수는 항상 현실의 정답 확률을 의미하지는 않습니다.")


## 7.4. Generative AI & Large Language Models

공개 소형 영어 언어 모델을 불러와 Prompt에 따른 텍스트 생성. 최초 모델 다운로드 필요.

In [ ]:
"""7.4: Hugging Face 공개 소형 언어 모델로 텍스트 생성.

모델: HuggingFaceTB/SmolLM2-135M-Instruct (주로 영어). 첫 실행 인터넷 필요.
작은 모델이므로 일반 상용 AI 수준의 정확도/품질을 기대하면 안 됨.
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
PROMPT = "List three simple ways to save electricity at home. Keep each tip short."
MAX_NEW_TOKENS = 70

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("모델을 처음 실행하는 경우 가중치를 다운로드합니다.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
model.eval()

messages = [{"role": "user", "content": PROMPT}]
# 역할 정보 등을 모델이 기대하는 입력 형식으로 변환
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt"
).to(device)
print("Input token count:", input_ids.shape[-1])

with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,             # 재현하기 쉬운 탐욕적 생성
        pad_token_id=tokenizer.eos_token_id,
    )

# 입력 문장 제외, 새롭게 생성된 토큰만 디코딩
new_tokens = output_ids[0][input_ids.shape[-1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("\n[Prompt]", PROMPT)
print("\n[Generated Text]\n", answer.strip())
print("\n주의: 생성 결과는 검증되지 않은 문장일 수 있습니다.")


## Exercise

- **Exercise 01:** `EPOCHS`와 `HIDDEN_UNITS`를 변경하여 학습 Loss와 테스트 정확도 비교.
- **Exercise 02:** 7.3에서 다른 공개 사물 사진을 업로드한 뒤 Top 5 결과 비교.
- **Exercise 03:** 7.4의 `PROMPT`와 `MAX_NEW_TOKENS`를 변경하여 문장 길이·형식을 비교.

세부 문제와 표는 [README.md](README.md)의 `Exercise` 참조.